# Viterbi HMM Decoding Assignment

## Viterbi Decoding Assignment

In this assignment, you will get a single DNA sequence in which subsequences from the human genome have been randomly concatenated with subsequences from the genome of the malaria parasite, which has a much higher average percentage of As and Ts than human. You will use a simple, two-state HMM based on the difference in frequencies of the nucleotides to try to figure out which segments come from human and which from malaria. To do this, you will implement the Viterbi algorithm (Part 1). In Part 2, you will manually tweak the HMM we gave you for Part 1 to try to improve the classification accuracy.

## Part 1: Coding the Viterbi Algorithm

There are unit tests for all the functions you have to write in the Test directory. `test_assignment.py` has a lot of tests using very small input sequences and HMMs designed to test certain expected behaviors. Make sure you can pass all these tests before going on to the large tests. The large tests check the number of correct predictions on a the large file of concatenated human and malaria sequences. Your code must pass all the tests.

### Input, Output, and Data Structures

1. Read in the merged human and malaria sequence `mixed2.fa` with `HMM.read_fasta()` (code provided in the cse587Autils package). `read_fasta()` reads a fasta file and outputs the nucleotide sequence with A, C, G, T converted to 0, 1, 2, 3. For example:

In [69]:
import os
from cse587Autils.HMMObjects.HMM import HMM
from assignment import viterbi_decode

# Get the path to the Data directory
DATA_DIR = os.path.join(os.getcwd(), "Data")

observations = HMM.read_fasta(os.path.join(DATA_DIR, "veryShortFasta.fa"))
print(observations)

[[0, 3, 2, 0, 3, 3, 3, 2, 2, 1, 2, 1, 1, 2, 1]]


Note that fasta files can contain multiple sequences, which is why `read_fasta()` returns a list of lists. For this assignment, we use only the first sequence in the file -- if there are others, they are ignored.

2. Read in the HMM with `HMM.read_hmm_file()` (code provided in the cse587Autils package). `read_hmm_file()` takes a text file representing an HMM and returns an HMM object.

Here's what a call to `HMM.read_hmm_file()` would look like:

In [70]:
hmm_object = HMM.read_hmm_file(os.path.join(DATA_DIR, "humanMalaria.hmm"))
hmm_object

HMM(states=['M', 'H'], num_states=2, alphabet=['A', 'C', 'G', 'T'], num_alphabet_symbols=4)

We have provided a function for checking the validity of an HMM object and diagnosing any problems with it. It is a good idea to run this check each time you input an HMM.

In [71]:
hmm_object.check_validity()

True

You can access the component parts of the HMM object like this:

In [72]:
hmm_object.states

['M', 'H']

In [73]:
hmm_object.initial_state_probs

[0.5, 0.5]

In [74]:
hmm_object.transition_matrix
# Each inner list (matrix row) gives the transition probabilities out of one state.
# If you need the transition probabilities into a state, you'll either need to transpose
# the array and take a row or access a column.

[[0.5, 0.5], [0.5, 0.5]]

In [75]:
hmm_object.alphabet

['A', 'C', 'G', 'T']

In [76]:
hmm_object.emission_matrix

[[0.3, 0.25], [0.2, 0.25], [0.2, 0.25], [0.3, 0.25]]

**Note** how the emission matrix is now transposed, relative to the HMM file, so rows correspond to alphabet letters and columns to states. This makes the coding simpler.

3. Implement the Viterbi and traceback algorithm by writing the function `viterbi_decode()`. The code file `assignment.py` contains stubs for 3 functions:

- `_build_matrix(observation_seq, hmm)` takes a list of integers corresponding to the observation sequence and an HMM object and returns the Viterbi matrix. Note that the matrix is the transpose of the way it was shown in class -- "rows" (the inner lists) correspond to observations and columns to states. This makes it much easier to implement.

In [83]:
from assignment import _build_matrix

matrix1 = _build_matrix(
    HMM.read_fasta(os.path.join(DATA_DIR, "veryShortFasta.fa"))[0],
    HMM.read_hmm_file(os.path.join(DATA_DIR, "humanMalaria.hmm"))
)
matrix1

array([[0.54545455, 0.45454545],
       [0.54545455, 0.45454545],
       [0.44444444, 0.55555556],
       [0.54545455, 0.45454545],
       [0.54545455, 0.45454545],
       [0.54545455, 0.45454545],
       [0.54545455, 0.45454545],
       [0.44444444, 0.55555556],
       [0.44444444, 0.55555556],
       [0.44444444, 0.55555556],
       [0.44444444, 0.55555556],
       [0.44444444, 0.55555556],
       [0.44444444, 0.55555556],
       [0.44444444, 0.55555556],
       [0.44444444, 0.55555556]])

- `_traceback(viterbi_matrix, hmm)` takes the output of `_build_matrix` and an HMM object and returns the most likely sequence of states, represented as a list of state numbers.

In [81]:
from assignment import _traceback

_traceback(
    matrix1,
    HMM.read_hmm_file(os.path.join(DATA_DIR, "humanMalaria.hmm"))
)

[0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1]

- `viterbi_decode(observation_seq, hmm)` takes a sequence of observations and an HMM object and returns the most likely sequence of states, represented as a list of state names.

In [82]:
viterbi_decode(
    HMM.read_fasta(os.path.join(DATA_DIR, "veryShortFasta.fa"))[0],
    HMM.read_hmm_file(os.path.join(DATA_DIR, "humanMalaria.hmm"))
)

['M', 'M', 'H', 'M', 'M', 'M', 'M', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H']

##### A few hints:

- If you're not sure where to start, read pages 3-6 of the HMM class notes (HMMnotes.pdf). Work through a toy example like you did with the at-home exercise: given an HMM and short observation sequence, e.g., ACC, compute the Viterbi dynamic programming table. Then, find the maximum cell in the final column of the table. Trace back to the source of the max in the previous column.

- `_build_matrix` goes through the observation sequence, from first to last, calculating the Viterbi probabilities. As soon as the probabilities are calculated for a given observation, you should normalize them by dividing each entry by the total over all entries. The results are not strictly "Viterbi" probabilities, but they are proportional to the Viterbi probabilities within each observation, and that's all that matters for the traceback. If you don't normalize, the numbers get very small very fast (in fact, exponentially fast) and you risk numerical underflow.

- There is no need to store traceback pointers on the forward pass. `_traceback` starts from the end of the Viterbi matrix with the highest scoring state for the last observation. It then calculates the Viterbi state for the second to last observation by using the Viterbi probabilities for the second to last observation and the transition probabilities (the emissions don't matter here). And so on backward until the first observation is reached.

- The most straightforward way to write `_build_matrix` is with two `for` loops, one for the observation index and one for the state index. However, if you want an extra challenge and a more elegant implementation, try doing without a state index. You can do this by using vector operations. Hint: You do not actually need dot products or matrix products for Viterbi. Instead, use element-wise multiplication on vectors and matrices. You may also find `np.max`, `np.sum`, and `np.transpose` useful.

- The Viterbi algorithm does not specify which trace back path to take in the case of ties. However, we have provided a function `_max_position` that takes an array and provides the index of the maximum. It resolves ties in favor of the earlier of the tied values. It also treats values that differ by less than a factor of 1E-5 as ties. This is important for dealing with numerical issues and insuring that your results match the expected results.

4. Evaluate the accuracy of the HMM with `calculate_accuracy()` (Code provided in the cse587Autils package). This function takes the state sequence you generated with `viterbi_decode()` and calculates the number of correctly labeled states. You should expect some bases to be misattributed, particularly near the transitions from one state to another.

Here's what a call to `calculate_accuracy()` would look like:

In [67]:
from cse587Autils.HMMObjects.HMM import calculate_accuracy

calculate_accuracy(
    viterbi_decode(
        HMM.read_fasta(os.path.join(DATA_DIR, "mixed2.fa"))[0],
        HMM.read_hmm_file(os.path.join(DATA_DIR, "humanMalaria.hmm"))
    ),
    HMM.read_fasta(os.path.join(DATA_DIR, "mixed2key.fa"))[0]
)

118389

The fraction correct can be calculated as

In [68]:
118389 / len(HMM.read_fasta(os.path.join(DATA_DIR, "mixed2.fa"))[0])

0.6743160808570989

## Part 2: Playing Around with the HMM Parameters

67% correct is pretty good, given that the random expectation is only 50%. But if you look at the HMM we gave you, you will see that it uses very round numbers that can't possibly be accurate. Play around with the parameters. Can you improve accuracy of the model? Please make a new .hmm file in the assignment directory called "tweakedHMM.hmm". In the cells below, run it on `mixed2.fa` and calculate the fraction of correctly labeled nucleotides, as was done at the end of Part 1. Below that, please offer a brief comment on the accuracy improvement (if any) and what you did to achieve it.

Some things to think about as you do this:

1. What are the true GC and AT percentages in the human and malaria genomes? You may be able to find this by Googling. If you're feeling ambitious, you can calculate the percent of each nucleotide in the human and malaria sequences we provided for you, by splitting the sequence into its human and malaria segments using `mixed2key.fa`.

2. What do you think the length of the contiguous segments of human and malaria are in `mixed2.fa`? You don't really have a good way to guess, but think about the average implied by the switching frequency in the HMM we gave you. Does that seem too big or too small? If you're feeling even a little ambitious, you can calculate it from `mixed2key.fa`.